In [1]:
from transformers import Trainer, TrainingArguments, DistilBertForSequenceClassification, DistilBertTokenizerFast, \
     DataCollatorWithPadding, pipeline
from datasets import load_metric, Dataset
import numpy as np
from sklearn.preprocessing import LabelEncoder

C:\ProgramData\anaconda3\envs\llms\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import TensorDataset, DataLoader

In [3]:
normal = pd.read_csv('data/normal.csv')
anomalous = pd.read_csv('data/anomalous.csv')
data = pd.concat([normal, anomalous], ignore_index = True)
#data = data.sample(n=1000).reset_index()
print(data.columns)
data.head(2)

Index(['class', 'Method', 'User-Agent', 'Pragma', 'Cache-Control', 'Accept',
       'Accept-encoding', 'Accept-charset', 'language', 'host', 'cookie',
       'content-type', 'connection', 'lenght', 'content', 'classification',
       'URL'],
      dtype='object')


,class,Method,User-Agent,Pragma,Cache-Control,Accept,Accept-encoding,Accept-charset,language,host,cookie,content-type,connection,lenght,content,classification,URL
0,Normal,GET,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,JSESSIONID=1F767F17239C9B670A39E9B10C3825F4,NaN,close,NaN,NaN,0,http://localhost:8080/tienda1/index.jsp HTTP/1.1
1,Normal,GET,Mozilla/5.0 (compatible; Konqueror/3.5; Linux)...,no-cache,no-cache,"text/xml,application/xml,application/xhtml+xml...","x-gzip, x-deflate, gzip, deflate","utf-8, utf-8;q=0.5, *;q=0.5",en,localhost:8080,JSESSIONID=81761ACA043B0E6014CA42A4BCD06AB5,NaN,close,NaN,NaN,0,http://localhost:8080/tienda1/publico/anadir.j...


In [4]:
include =['object', 'float', 'int']
#data.describe(include=include)

In [5]:
def make_http_string(datset):
  #X = datset[['Method','User-Agent','Pragma','Cache-Control', 'Accept','Accept-encoding','language', 'host', 'cookie', 'content-type', 'connection','lenght', 'content','URL']]
  X = datset[['URL', 'content']]
  y = datset['classification']
  X_string = []
  cols = X.columns
  for i in range(len(X)):
      temp = ''
      for col in cols:
          temp += str(X.iloc[i][col])
      ans = ''
      for t in temp:
          if t.isalpha():
              ans += t.lower()
          else:
              ans += t
      X_string.append(ans)
  return X_string
y = data['classification']
X_string = make_http_string(data)

In [6]:
dataDict = dict(
    sentence=X_string,
    label=y
)
print(len(dataDict))

2


In [7]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-cased')

In [8]:
ds = Dataset.from_dict(dataDict)
ds = ds.train_test_split(test_size=0.2)

In [15]:
ds["train"][3656]

{'sentence': 'http://localhost:8080/tienda1/publico/registro.jsp?modo=registro&login=dorrie&password=acopio&nombre=urban%27o&apellidos=buix%f3+diclo&email=ben-amotz-hehn%40clubforex.io&dni=53702702w&direccion=pla%e7a+josep+maria+folch+i+torres%2c+15+5-g&ciudad=villamandos&cp=09244&provincia=badajoz&ntc=1143578341526495&b1=registrar http/1.1nan',
 'label': 1}

In [11]:
# simple function to batch tokenize utterances with truncation
def preprocess_function(examples):
    return tokenizer(examples["sentence"], truncation=True)

In [12]:
seq_clf_tokenized_sents = ds.map(preprocess_function, batched=True)

Map: 100%|██████████| 12213/12213 [00:01<00:00, 6374.34 examples/s]


In [13]:
seq_clf_tokenized_sents["train"][0]

{'sentence': 'http://localhost:8080/tienda1/imagenes/nuestratierra.jpg http/1.1nan',
 'label': 0,
 'input_ids': [101,
  8413,
  131,
  120,
  120,
  1469,
  15342,
  1204,
  131,
  2908,
  18910,
  120,
  5069,
  7807,
  1475,
  120,
  3077,
  3965,
  120,
  183,
  27648,
  7625,
  2852,
  1611,
  119,
  179,
  1643,
  1403,
  8413,
  120,
  122,
  119,
  122,
  6509,
  102],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

In [14]:
# DataCollatorWithPadding creates batch of data. It also dynamically pads text to the 
#  length of the longest element in the batch, making them all the same length. 
#  It's possible to pad your text in the tokenizer function with padding=True, dynamic padding is more efficient.

# Data Collator will pad data so that all examples are the same input length.
#  Attention mask is how we ignore attention scores for padding tokens
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [15]:
unique_sequence_labels = [0, 1]

In [16]:
sequence_clf_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-cased', 
    num_labels=len(unique_sequence_labels),
)

# set an index -> label dictionary
sequence_clf_model.config.id2label = {i: l for i, l in enumerate(unique_sequence_labels)}

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
sequence_clf_model.config.id2label[0]

0

In [18]:
metric = load_metric("accuracy")

def compute_metrics(eval_pred):  # custom method to take in logits and calculate accuracy of the eval set
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


C:\Users\Hassan\AppData\Local\Temp\ipykernel_3752\2176569874.py:1: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric("accuracy")
C:\ProgramData\anaconda3\envs\llms\Lib\site-packages\datasets\load.py:759: FutureWarning: The repository for accuracy contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.1/metrics/accuracy/accuracy.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


In [19]:
epochs = 2

training_args = TrainingArguments(
    output_dir="./sents_clf/results",
    num_train_epochs=epochs,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    load_best_model_at_end=True,
    
    # some deep learning parameters that the Trainer is able to take in
    warmup_steps=len(seq_clf_tokenized_sents['train']) // 5,  # number of warmup steps for learning rate scheduler,
    weight_decay = 0.05,
    
    logging_steps=1,
    log_level='info',
    evaluation_strategy='epoch',
    eval_steps=50,
    save_strategy='epoch'
)


C:\Users\Hassan\AppData\Roaming\Python\Python312\site-packages\transformers\training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [20]:

# Define the trainer:

trainer = Trainer(
    model=sequence_clf_model,
    args=training_args,
    train_dataset=seq_clf_tokenized_sents['train'],
    eval_dataset=seq_clf_tokenized_sents['test'],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

In [41]:
# Get initial metrics
trainer.evaluate()

The following columns in the evaluation set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: sentence. If sentence are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 12213
  Batch size = 32


{'eval_loss': 0.6829990148544312,
 'eval_model_preparation_time': 0.0026,
 'eval_accuracy': 0.5814296241709653,
 'eval_runtime': 11128.7121,
 'eval_samples_per_second': 1.097,
 'eval_steps_per_second': 0.034}

In [21]:
trainer.train(resume_from_checkpoint=True)

Loading model from ./sents_clf/results\checkpoint-3054.
The following columns in the training set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: sentence. If sentence are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 48,852
  Num Epochs = 2
  Instantaneous batch size per device = 32
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 1
  Total optimization steps = 3,054
  Number of trainable parameters = 65,783,042
  Continuing training from checkpoint, will skip to saved global_step
  Continuing training from epoch 2
  Continuing training from global step 3054
  Will skip the first 2 epochs then the first 0 batches in the first epoch.


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from ./sents_clf/results\checkpoint-30

Epoch,Training Loss,Validation Loss


TrainOutput(global_step=3054, training_loss=0.0, metrics={'train_runtime': 0.2743, 'train_samples_per_second': 356180.508, 'train_steps_per_second': 11133.375, 'total_flos': 5384449958641824.0, 'train_loss': 0.0, 'epoch': 2.0})

In [22]:
trainer.evaluate()

The following columns in the evaluation set don't have a corresponding argument in `DistilBertForSequenceClassification.forward` and have been ignored: sentence. If sentence are not expected by `DistilBertForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 12213
  Batch size = 32


{'eval_loss': 0.04310423508286476,
 'eval_accuracy': 0.9887005649717514,
 'eval_runtime': 4628.5734,
 'eval_samples_per_second': 2.639,
 'eval_steps_per_second': 0.083,
 'epoch': 2.0}

In [25]:
trainer.save_model()

Saving model checkpoint to ./sents_clf/results
Configuration saved in ./sents_clf/results\config.json
Model weights saved in ./sents_clf/results\model.safetensors
